# an effection way to load ham

In [1]:
import numpy as np
import jax
import jax.numpy as jnp
from jax import jit
from functools import lru_cache

In [2]:
jax.config.update("jax_enable_x64", True)

In [3]:
import h5py

# 你想查看的数据集名称，你可以随时改成 'ham_BK-8', 'ham_molec-4' 等等
dataset_name = 'ham_JW-4' 

with h5py.File('./ham/H2.hdf5', 'r') as f:
    dataset = f[dataset_name]
    data = dataset[()] # type: ignore
    
    print(f"=== {dataset_name} 的数据类型: {type(data)} ===")
    if isinstance(data, bytes):
        try:
            print(data.decode('utf-8'))
        except UnicodeDecodeError:
            print("数据是二进制字节，可能使用了 pickle 等方式序列化。")
            print(data)
    else:
        print(data)

=== ham_JW-4 的数据类型: <class 'bytes'> ===
(-0.44779757933958947+0j) [] +
(-0.014034099995004047+0j) [X0 X1 Y2 Y3] +
(0.014034099995004047+0j) [X0 Y1 Y2 X3] +
(0.014034099995004047+0j) [Y0 X1 X2 Y3] +
(-0.014034099995004047+0j) [Y0 Y1 X2 X3] +
(0.2823508885110738+0j) [Z0] +
(0.16462320552994025+0j) [Z0 Z1] +
(0.08211612483661979+0j) [Z0 Z2] +
(0.09615022483162383+0j) [Z0 Z3] +
(0.28235088851107354+0j) [Z1] +
(0.09615022483162383+0j) [Z1 Z2] +
(0.08211612483661979+0j) [Z1 Z3] +
(-0.003986747692442089+0j) [Z2] +
(0.08366738652351666+0j) [Z2 Z3] +
(-0.0039867476924421025+0j) [Z3]


In [4]:
pauli_mat = jnp.eye(4)
print(pauli_mat)

E0324 22:35:35.428390   39669 cuda_executor.cc:1743] Could not get kernel mode driver version: [INVALID_ARGUMENT: Version does not match the format X.Y.Z]
E0324 22:35:35.438369   39550 cuda_executor.cc:1743] Could not get kernel mode driver version: [INVALID_ARGUMENT: Version does not match the format X.Y.Z]


[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]


In [5]:
class Pauli:
        r"""Configuration manager for Pauli string observables.
            
        Examples
        --------
        >>> config = PauliStringConfig(n_q=4, n_p=2)
        >>> config.add(0, "ZZII", 0.5)  # Add 0.5 * Z⊗Z⊗I⊗I
        >>> config.add(1, "XIXY", -0.4) # Add -0.4 * X⊗I⊗X⊗Y
        """
        
        def __init__(self, n_q: int, n_p: int):
            self.n_q = n_q
            self.n_p = n_p
            self.mat = jnp.zeros((self.n_p, self.n_q, 4))
            self.weights = jnp.ones(self.n_p)
            self.sync()

        def sync(self):
            self.tn =  self.mat @ pauli_mat

        @property
        def w(self):
            return self.weights
        
        @w.setter
        def w(self, value):
            self.weights = value

        @property
        def data(self):
            r"""Return (tn, w) tuple."""
            return (self.tn, self.w)
        
        def add(self, index: int, pauli_string: str, weight: float):
                p_str = pauli_string.strip().upper()

                if len(p_str) != self.n_q:
                    raise ValueError(
                        f"Pauli string length ({len(p_str)}) does not match "
                        f"num_qubits ({self.n_q})."
                    )
                
                mapping = {'I': 0, 'X': 1, 'Y': 2, 'Z': 3}
                
                indices = []
                for char in p_str:
                    if char not in mapping:
                        raise ValueError(
                            f"Invalid character '{char}' in Pauli string. "
                            f"Only I, X, Y, Z are supported."
                        )
                    indices.append(mapping[char])
                
                row_values = jax.nn.one_hot(jnp.array(indices), 4)
            
                self.mat = self.mat.at[index].set(row_values)
                self.weights = self.weights.at[index].set(weight)
                self.sync()

        def save(self, path: str):
                """Save Pauli data to a .npz file."""
                np.savez(path, mat=np.asarray(self.mat), w=np.asarray(self.w))
        
        @classmethod
        def load(cls, path: str) -> 'Pauli':
                """Load Pauli data from a .npz file."""
                data = np.load(path)
                mat, w = data['mat'], data['w']
                n_p, n_q = mat.shape[0], mat.shape[1]
                obj = cls(n_q=n_q, n_p=n_p)
                obj.mat = jnp.array(mat)
                obj.w = jnp.array(w)
                obj.sync()
                return obj

In [6]:

import re

raw = data.decode('utf-8') if isinstance(data, bytes) else data

pattern = r'\(([^)]+)\)\s*\[([^\]]*)\]'
terms = []
for match in re.finditer(pattern, raw):
    coeff = float(match.group(1).replace('+0j', '').replace('+0', ''))
    ops_str = match.group(2).strip()
    
    if not ops_str:
        continue
    
    pauli_map = {}
    for op in ops_str.split():
        gate = op[0]
        qubit = int(op[1:])
        pauli_map[qubit] = gate
    
    pauli_str = ''
    for q in range(4):
        pauli_str += pauli_map.get(q, 'I')
    
    terms.append((pauli_str, coeff))
    
n_q = 4
n_p = len(terms)
ham = Pauli(n_q=n_q, n_p=n_p)

for i, (pstr, w) in enumerate(terms):
    ham.add(i, pstr, w)
    print(f"  [{i}] {pstr}  weight={w}")

print(f"\n共 {n_p} 个 Pauli 项 (已省略单位算符)")

  [0] XXYY  weight=-0.014034099995004047
  [1] XYYX  weight=0.014034099995004047
  [2] YXXY  weight=0.014034099995004047
  [3] YYXX  weight=-0.014034099995004047
  [4] ZIII  weight=0.2823508885110738
  [5] ZZII  weight=0.16462320552994025
  [6] ZIZI  weight=0.08211612483661979
  [7] ZIIZ  weight=0.09615022483162383
  [8] IZII  weight=0.28235088851107354
  [9] IZZI  weight=0.09615022483162383
  [10] IZIZ  weight=0.08211612483661979
  [11] IIZI  weight=-0.003986747692442089
  [12] IIZZ  weight=0.08366738652351666
  [13] IIIZ  weight=-0.0039867476924421025

共 14 个 Pauli 项 (已省略单位算符)


In [16]:
import re
import h5py

def load_pauli_from_hdf5(filepath: str, dataset_name: str) -> Pauli:
    """Load a Pauli Hamiltonian from an HDF5 file. Identity-only terms are skipped."""
    with h5py.File(filepath, 'r') as f:
        raw = f[dataset_name][()]
    if isinstance(raw, bytes):
        raw = raw.decode('utf-8')

    pattern = r'\(([^)]+)\)\s*\[([^\]]*)\]'
    terms = []
    for match in re.finditer(pattern, raw):
        coeff = float(match.group(1).replace('+0j', '').replace('-0j', ''))
        ops_str = match.group(2).strip()
        if not ops_str:
            continue

        pauli_map = {}
        n_q = 0
        for op in ops_str.split():
            qubit = int(op[1:])
            pauli_map[qubit] = op[0]
            n_q = max(n_q, qubit + 1)
        terms.append((pauli_map, coeff, n_q))

    n_q = max(t[2] for t in terms)
    ham = Pauli(n_q=n_q, n_p=len(terms))
    for i, (pauli_map, w, _) in enumerate(terms):
        pauli_str = ''.join(pauli_map.get(q, 'I') for q in range(n_q))
        ham.add(i, pauli_str, w)
    return ham

In [17]:
ham = load_pauli_from_hdf5(filepath='./ham/H2.hdf5', dataset_name='ham_JW-4')

In [18]:
print(ham.data[0].shape)

(14, 4, 4)


In [19]:
ham.save('ham_JW4.npz')

In [20]:
h = Pauli.load('ham_JW4.npz')

In [21]:
h.data[0].shape

(14, 4, 4)

In [22]:
h.data[0][0]

Array([[0., 1., 0., 0.],
       [0., 1., 0., 0.],
       [0., 0., 1., 0.],
       [0., 0., 1., 0.]], dtype=float64)

In [23]:
h.data[1]

Array([-0.0140341 ,  0.0140341 ,  0.0140341 , -0.0140341 ,  0.28235089,
        0.16462321,  0.08211612,  0.09615022,  0.28235089,  0.09615022,
        0.08211612, -0.00398675,  0.08366739, -0.00398675], dtype=float64)